In [ ]:
import glob
import networkx as nx
import altair as alt
import pandas as pd
from collections import Counter
from statistics import stdev

In [ ]:
path = 'Data/simulated_dags_seed42/'
subdir = ['tracerx_128/',
          'tracerx_256/',
          'tracerx_512/',
          'tracerx_1024/',
          'tracerx_2048/'
          ]

In [ ]:
sim_stats = dict()

In [ ]:
for dir in subdir:
    edge_nums = []
    node_nums = []
    graph_count = 0
    for graph_file in glob.glob(path + dir + '*.gexf'):
        graph = nx.read_gexf(graph_file)
        graph = nx.transitive_closure(graph)
        edge_nums.append(len(graph.edges))
        node_nums.append(len(graph.nodes))
        graph_count += 1
    edge_dev = stdev(edge_nums)
    edge_mean = sum(edge_nums) / graph_count
    min_edges = min(edge_nums)
    max_edges = max(edge_nums)

    node_dev = stdev(node_nums)
    node_mean = sum(node_nums) / graph_count
    min_nodes = min(node_nums)
    max_nodes = max(node_nums)

    sim_stats[dir] = {'edge_dev': edge_dev, 'edge_mean': edge_mean, 'min_edges': min_edges, 'max_edges': max_edges, 'node_mean': node_mean, 'node_dev': node_dev, 'node_min': min_nodes, 'node_max': max_nodes}

In [ ]:
df = pd.DataFrame.from_dict(sim_stats, orient='index')
df.loc['mean'] = df.mean()
df

In [ ]:
result_file = '/Output/benchmarking/new_benchmarking_runtime/average_runtimes_and_peak_mem.csv'
df_runtimes = pd.read_csv(result_file)
df_runtimes


In [ ]:
df_runtimes['maximum resident set size'] = df_runtimes['maximum resident set size'] / 1024 # convert MB to GB

In [ ]:
wall = alt.Chart(df_runtimes).mark_line(point=True, color="#98b2f3").encode(
    x=alt.X('Graph count', ),
    y=alt.Y('wall-clock time', axis=alt.Axis(titleColor='#98b2f3',
                                             labelFontSize=16,
                                             titleFontSize=16,
                                             title='Wall-clock time (sec)'
                                             )),
)

rss = alt.Chart(df_runtimes).mark_line(point=True, color="#cb2a7b").encode(
    x=alt.X('Graph count', axis=alt.Axis(labelFontSize=16, titleFontSize=16)),
    y=alt.Y('maximum resident set size', axis=alt.Axis(titleColor='#cb2a7b',
                                                       labelFontSize=16,
                                                       titleFontSize=16,
                                                       title='Maximum RSS (GB)')),
    color=alt.value("#cb2a7b")
)

chart = alt.layer(wall, rss).resolve_scale(
    y='independent'
).properties(
    title=alt.TitleParams(
        text='Effect of instance size on runtime and memory',
        fontSize=16
    )
)

chart.show()
chart.save('/Output/benchmarking results/effect_of_instance_size_on_runtime_and_memory.svg', 'svg')

In [ ]:
histo = alt.Chart(df_runtimes).mark_bar(color="#98b2f3").encode(
    x=alt.X('Graph count:O', axis=alt.Axis(labelFontSize=16, titleFontSize=16, labelAngle=0)),
    y=alt.Y('trajectory size', axis=alt.Axis(labelFontSize=16, titleFontSize=16, title='Trajectory size (nodes)')),
).properties(
    width=200,
    title=alt.TitleParams(
        text='Output size',
        fontSize=16
    )
)

histo.show()
histo.save('/Output/benchmarking results/output_sizes.svg', 'svg')